# Notebook 5: DBSCAN — Density-Based Spatial Clustering

**Difficulty:** ⭐⭐⭐⭐ | **Estimated Time:** 60 minutes

---

## Learning Objectives

By the end of this notebook you will be able to:

1. Distinguish between **core points**, **border points**, and **noise points**
2. Implement DBSCAN from scratch using region queries
3. Explain how the parameters **ε** (epsilon) and **min_samples** control cluster shape and noise tolerance
4. Demonstrate that DBSCAN handles non-convex clusters and outliers that K-means cannot

---

**Week 11 Theme:** Customer segmentation / retail data


## The Intuition: Towns on a Satellite Map

Imagine you are looking at a satellite image of a country and you want to identify towns — without anyone telling you in advance how many towns there are.

Here is how you would do it intuitively:

> Pick a random house. If it has at least **min_people** neighbouring houses within a radius of **ε** kilometres, declare it a **town centre** (core point). Now spread outward: any house within ε of a town centre also belongs to that town. Houses that are within ε of *someone* in the town but not dense enough themselves are on the **outskirts** (border points). Houses too isolated to be connected to any dense cluster are marked **noise** — perhaps a lone farmhouse in the middle of nowhere.

This is exactly DBSCAN:
- **ε (epsilon):** the neighbourhood radius
- **min_samples:** the minimum number of neighbours needed to declare a core point
- Clusters grow organically by connecting dense neighbourhoods — they can be any shape
- Isolated points are automatically labelled as outliers (noise)


## Why Does This Matter for ML?

Most real-world clusters are **not spherical**. Consider:

- Customer journeys that form winding paths through feature space
- Geographic regions that follow coastlines or river systems
- Anomaly detection: fraudulent transactions are genuinely isolated outliers

**K-means fails** on all of these because it assumes every cluster is a round blob around a centroid.

DBSCAN offers three key advantages over K-means:

| Advantage | Description |
|-----------|-------------|
| **Arbitrary shapes** | Clusters can be crescent-shaped, ring-shaped, elongated, or any shape |
| **Automatic outlier detection** | Noise points are labelled -1 without any extra step |
| **No k needed** | You never have to specify the number of clusters in advance |

In retail, DBSCAN can find unusual spending patterns (noise = potential VIP or fraud) and geographic customer clusters without assuming circular trade areas.


In [ ]:
# ── Core imports ──────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from sklearn.datasets import make_moons, make_circles, make_blobs
from sklearn.cluster import KMeans, DBSCAN as SklearnDBSCAN
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

np.random.seed(42)            # reproducibility
sns.set_theme(style='whitegrid')  # consistent plot style

print("Imports complete.")


## The Three Point Types — Formal Definitions

Before writing any code, let us pin down the three categories precisely.

Given a dataset **X**, a radius **ε > 0**, and a threshold **min_samples ≥ 1**:

### 1. Core Point

Plain English: *A point that is in the middle of a dense region — it has enough neighbours nearby.*

Formally: Point **p** is a core point if
$$|N_\varepsilon(p)| \geq \text{min\_samples}$$
where $N_\varepsilon(p) = \{q \in X : d(p, q) \leq \varepsilon\}$ is the **ε-neighbourhood** of p (which includes p itself).

### 2. Border Point

Plain English: *A point that is close to a core point but not dense enough to be a core itself — it sits on the edge of a cluster.*

Formally: Point **p** is a border point if:
- $|N_\varepsilon(p)| < \text{min\_samples}$ (not a core point), **AND**
- $\exists$ a core point $q$ such that $d(p, q) \leq \varepsilon$

### 3. Noise Point (Outlier)

Plain English: *A point that is neither core nor border — it is isolated, belonging to no cluster.*

Formally: Point **p** is noise if it is neither a core point nor a border point.

---

**Key property:** Two core points that are within ε of each other belong to the same cluster. A border point inherits the cluster label of its nearest core point.


In [ ]:
np.random.seed(42)

# ── Build a hand-crafted 2D dataset to illustrate point types ─────────────────
# Dense cluster A
cluster_a = np.random.randn(20, 2) * 0.3 + [0, 0]
# Dense cluster B
cluster_b = np.random.randn(20, 2) * 0.3 + [3, 1]
# A few isolated points (noise candidates)
noise_pts = np.array([[1.5, 3.0], [2.0, -1.5], [-1.0, 1.8]])

X_demo = np.vstack([cluster_a, cluster_b, noise_pts])

# Parameters for illustration
EPS_DEMO = 0.6
MIN_SAMPLES_DEMO = 4

# ── Classify each point ───────────────────────────────────────────────────────
def classify_points(X, eps, min_samples):
    """Return list of labels: 'core', 'border', or 'noise' for each point."""
    n = len(X)
    # Count neighbours within eps (including self)
    neighbour_counts = np.array([
        np.sum(np.linalg.norm(X - X[i], axis=1) <= eps)
        for i in range(n)
    ])
    is_core = neighbour_counts >= min_samples

    labels = []
    for i in range(n):
        if is_core[i]:
            labels.append('core')
        else:
            # Border if within eps of ANY core point
            dists_to_cores = np.linalg.norm(X[is_core] - X[i], axis=1)
            if len(dists_to_cores) > 0 and dists_to_cores.min() <= eps:
                labels.append('border')
            else:
                labels.append('noise')
    return labels

point_labels = classify_points(X_demo, EPS_DEMO, MIN_SAMPLES_DEMO)

# ── Plot ──────────────────────────────────────────────────────────────────────
color_map = {'core': 'steelblue', 'border': 'orange', 'noise': 'red'}
marker_map = {'core': 'o', 'border': 's', 'noise': 'X'}

fig, ax = plt.subplots(figsize=(8, 6))

for lbl in ['core', 'border', 'noise']:
    mask = [p == lbl for p in point_labels]
    ax.scatter(
        X_demo[mask, 0], X_demo[mask, 1],
        c=color_map[lbl], marker=marker_map[lbl],
        s=80, zorder=3, label=lbl.capitalize()
    )

# Draw eps circles around one core point and one noise point as examples
example_core_idx = 0   # first point in cluster A (likely a core)
example_noise_idx = len(X_demo) - 1  # last noise point

for idx, color, lw in [(example_core_idx, 'steelblue', 2),
                        (example_noise_idx, 'red', 2)]:
    circle = plt.Circle(
        X_demo[idx], EPS_DEMO,
        color=color, fill=False, linestyle='--', linewidth=lw, alpha=0.7
    )
    ax.add_patch(circle)
    ax.annotate(
        f'ε={EPS_DEMO}', xy=X_demo[idx],
        xytext=(X_demo[idx][0] + EPS_DEMO + 0.05, X_demo[idx][1]),
        fontsize=9, color=color
    )

ax.set_title(f'DBSCAN Point Types  (ε={EPS_DEMO}, min_samples={MIN_SAMPLES_DEMO})', fontsize=13)
ax.set_xlabel('Feature 1')
ax.set_ylabel('Feature 2')
ax.legend()
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

from collections import Counter
print("Point type counts:", Counter(point_labels))


## The DBSCAN Algorithm — How Clusters Grow

Think of DBSCAN as a **spreading fire**:

1. Pick any unvisited point **p**.
2. Find all points within distance ε of p — its **ε-neighbourhood**.
3. **If** the neighbourhood has fewer than `min_samples` points → mark p as noise (for now) and move on.
4. **Otherwise** → p is a core point. Start a new cluster and **expand** it:
   - Label p with the new cluster ID.
   - For every neighbour q of p:
     - If q is unvisited, add q to the cluster.
     - If q is *also* a core point, add all of q's neighbours to the frontier (fire spreads).
5. Repeat until no unvisited points remain.

**Density-connectivity:** Two points belong to the same cluster if and only if there exists a chain of core points linking them, each within ε of the next. This is what allows DBSCAN to find winding, non-convex shapes.

> A noise point at first pass may be later claimed as a border point when a nearby cluster expands. But it will never become a core point.


In [ ]:
np.random.seed(42)

def dbscan_scratch(X, eps=0.5, min_samples=5):
    """
    DBSCAN from scratch.

    Parameters
    ----------
    X           : ndarray of shape (n, d)
    eps         : neighbourhood radius
    min_samples : minimum neighbours to be a core point (including self)

    Returns
    -------
    labels : ndarray of shape (n,)
        Cluster IDs starting at 0; -1 means noise.
    """
    n = len(X)
    labels = np.full(n, -1)   # -1 = unvisited / unassigned
    cluster_id = 0

    def region_query(point_idx):
        """Return indices of all points within eps of point_idx."""
        dists = np.linalg.norm(X - X[point_idx], axis=1)  # distance to every other point
        return list(np.where(dists <= eps)[0])             # indices within eps

    def expand_cluster(point_idx, neighbours, cluster_id):
        """Grow the cluster by exploring the frontier of dense neighbours."""
        labels[point_idx] = cluster_id   # assign seed point
        i = 0
        while i < len(neighbours):        # frontier grows as we iterate
            q = neighbours[i]
            if labels[q] == -1:           # q is unvisited
                labels[q] = cluster_id   # claim q for this cluster
                q_neighbours = region_query(q)
                if len(q_neighbours) >= min_samples:   # q is also a core point
                    neighbours.extend(q_neighbours)    # add q's neighbourhood to frontier
            # If labels[q] >= 0: already assigned — skip (may reassign border points)
            i += 1

    visited = np.zeros(n, dtype=bool)   # track which points we have processed

    for p in range(n):
        if visited[p]:       # already processed
            continue
        visited[p] = True
        neighbours = region_query(p)

        if len(neighbours) < min_samples:
            labels[p] = -2   # tentative noise (use -2 to distinguish from unvisited -1)
        else:
            expand_cluster(p, neighbours, cluster_id)
            cluster_id += 1  # next cluster gets a new ID

    labels[labels == -2] = -1   # convert tentative noise to official noise label
    return labels


# Quick sanity check on the demo dataset
demo_labels = dbscan_scratch(X_demo, eps=EPS_DEMO, min_samples=MIN_SAMPLES_DEMO)
print("Scratch labels (unique):", np.unique(demo_labels))
print("Cluster sizes:", {lbl: int((demo_labels == lbl).sum()) for lbl in np.unique(demo_labels)})


In [ ]:
np.random.seed(42)

# ── Generate non-convex datasets ──────────────────────────────────────────────
X_moons, _ = make_moons(n_samples=300, noise=0.07, random_state=42)
X_circles, _ = make_circles(n_samples=300, noise=0.05, factor=0.5, random_state=42)

# ── Run DBSCAN (scratch) and KMeans on both datasets ─────────────────────────
dbscan_moons   = dbscan_scratch(X_moons,   eps=0.25, min_samples=5)
dbscan_circles = dbscan_scratch(X_circles, eps=0.20, min_samples=5)

km_moons   = KMeans(n_clusters=2, random_state=42, n_init=10).fit_predict(X_moons)
km_circles = KMeans(n_clusters=2, random_state=42, n_init=10).fit_predict(X_circles)

# ── Plot: 2 datasets × 2 algorithms = 4 subplots ─────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
titles = [
    ('make_moons  — KMeans (fails)', km_moons,   X_moons),
    ('make_moons  — DBSCAN (wins)', dbscan_moons,  X_moons),
    ('make_circles — KMeans (fails)', km_circles, X_circles),
    ('make_circles — DBSCAN (wins)', dbscan_circles, X_circles),
]

palette = ['#4C72B0', '#DD8452', '#55A868', '#C44E52', '#8172B2']

for ax, (title, labels, X) in zip(axes.ravel(), titles):
    unique_labels = np.unique(labels)
    for lbl in unique_labels:
        mask = labels == lbl
        color = 'grey' if lbl == -1 else palette[lbl % len(palette)]
        marker = 'x' if lbl == -1 else 'o'
        ax.scatter(X[mask, 0], X[mask, 1],
                   c=color, marker=marker, s=25, alpha=0.7,
                   label=f'Noise' if lbl == -1 else f'Cluster {lbl}')
    ax.set_title(title, fontsize=11)
    ax.legend(fontsize=8, markerscale=1.2)

fig.suptitle('DBSCAN vs. KMeans on Non-Convex Data', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
np.random.seed(42)

# ── Detailed view: noise points as grey X markers ─────────────────────────────
# We use make_moons with slightly higher noise to get more outliers
X_noisy, _ = make_moons(n_samples=350, noise=0.12, random_state=42)
labels_noisy = dbscan_scratch(X_noisy, eps=0.25, min_samples=5)

fig, ax = plt.subplots(figsize=(8, 5))

cluster_colors = ['#4C72B0', '#DD8452']

# Plot each cluster
for lbl in np.unique(labels_noisy):
    if lbl == -1:
        continue   # handle noise separately
    mask = labels_noisy == lbl
    ax.scatter(X_noisy[mask, 0], X_noisy[mask, 1],
               c=cluster_colors[lbl % 2], s=40, alpha=0.8,
               label=f'Cluster {lbl}')

# Plot noise points as grey X markers
noise_mask = labels_noisy == -1
ax.scatter(X_noisy[noise_mask, 0], X_noisy[noise_mask, 1],
           c='dimgrey', marker='X', s=80, zorder=5,
           label=f'Noise ({noise_mask.sum()} points)')

ax.set_title('DBSCAN: Clusters (colored) and Noise (grey ✕)', fontsize=13)
ax.set_xlabel('Feature 1')
ax.set_ylabel('Feature 2')
ax.legend()
plt.tight_layout()
plt.show()

n_clusters = len(set(labels_noisy)) - (1 if -1 in labels_noisy else 0)
print(f"Clusters found: {n_clusters}")
print(f"Noise points  : {noise_mask.sum()} ({100*noise_mask.mean():.1f}%)")


## Choosing ε and min_samples

DBSCAN has two hyperparameters. Here are practical rules of thumb:

### min_samples
- Start with `min_samples ≥ 2 × n_features` (e.g., 4 for 2D data)
- Larger values → more points become noise, smoother cluster boundaries
- For small or noisy datasets, `min_samples = 5` is a safe default

### ε (epsilon) — The k-Distance Graph Method

The most principled way to pick ε:

1. Set `k = min_samples − 1`
2. For each point, compute the distance to its **k-th nearest neighbour**
3. Sort these distances in ascending order and plot them
4. Look for the **elbow** (the sharp uptick) in the curve
5. The ε value at the elbow is your target

**Intuition:** Points inside a cluster have small k-distances (neighbours are nearby). Points in sparse regions have large k-distances. The elbow separates these two regimes — setting ε just above the elbow captures dense clusters while excluding sparse noise regions.


In [ ]:
np.random.seed(42)

# ── k-Distance Graph implementation ──────────────────────────────────────────
def k_distance_graph(X, k=4):
    """
    Compute the k-th nearest neighbour distance for every point in X.
    Returns the sorted distances (ascending) for plotting.
    """
    # NearestNeighbors finds the k+1 nearest (including the point itself)
    nbrs = NearestNeighbors(n_neighbors=k+1).fit(X)
    distances, _ = nbrs.kneighbors(X)   # shape: (n, k+1)
    kth_dist = distances[:, k]           # distance to the k-th neighbour (0-indexed, so index k)
    return np.sort(kth_dist)             # sort ascending for the elbow plot


# Use make_moons as the working dataset
X_work, _ = make_moons(n_samples=300, noise=0.07, random_state=42)
MIN_SAMPLES = 5
k = MIN_SAMPLES - 1  # k = 4

sorted_dists = k_distance_graph(X_work, k=k)

# ── Find elbow using maximum curvature heuristic ──────────────────────────────
# Simple approach: second derivative of the sorted curve
second_deriv = np.diff(np.diff(sorted_dists))
elbow_idx = np.argmax(second_deriv) + 1   # +1 to account for diff offset
eps_suggested = sorted_dists[elbow_idx]

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 4))

ax.plot(sorted_dists, color='steelblue', linewidth=2, label=f'{k}-NN distance (sorted)')
ax.axhline(y=eps_suggested, color='tomato', linestyle='--', linewidth=1.5,
           label=f'Elbow ≈ {eps_suggested:.3f}  →  suggested ε')
ax.axvline(x=elbow_idx, color='tomato', linestyle=':', alpha=0.6)
ax.scatter([elbow_idx], [eps_suggested], color='tomato', s=80, zorder=5)

ax.set_title(f'k-Distance Graph  (k = {k} = min_samples − 1)', fontsize=13)
ax.set_xlabel('Points sorted by k-th NN distance')
ax.set_ylabel('Distance to k-th neighbour')
ax.legend()
plt.tight_layout()
plt.show()

print(f"Suggested ε = {eps_suggested:.4f}")


In [ ]:
np.random.seed(42)

# ── Parameter sensitivity grid ────────────────────────────────────────────────
eps_values        = [0.1, 0.3, 0.5, 0.8, 1.2]
min_samples_values = [3, 5, 10]

n_clusters_grid = np.zeros((len(min_samples_values), len(eps_values)), dtype=int)
n_noise_grid    = np.zeros_like(n_clusters_grid)

for i, ms in enumerate(min_samples_values):
    for j, ep in enumerate(eps_values):
        lbs = dbscan_scratch(X_work, eps=ep, min_samples=ms)
        n_clusters_grid[i, j] = len(set(lbs) - {-1})   # number of non-noise clusters
        n_noise_grid[i, j]    = (lbs == -1).sum()       # number of noise points

# ── Plot heatmaps ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, data, title, cmap in zip(
    axes,
    [n_clusters_grid, n_noise_grid],
    ['Number of Clusters Found', 'Number of Noise Points'],
    ['YlOrRd', 'Blues']
):
    sns.heatmap(
        data,
        annot=True, fmt='d', cmap=cmap,
        xticklabels=eps_values,
        yticklabels=min_samples_values,
        ax=ax, linewidths=0.5
    )
    ax.set_title(title, fontsize=12)
    ax.set_xlabel('ε (epsilon)')
    ax.set_ylabel('min_samples')

fig.suptitle('DBSCAN Parameter Sensitivity (make_moons, n=300)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("Observation: large ε merges everything into 1 cluster; tiny ε makes everything noise.")


## Comparison: K-means vs. Hierarchical vs. DBSCAN

| Property | K-means | Hierarchical (Agglomerative) | DBSCAN |
|----------|---------|------------------------------|--------|
| **Cluster shape** | Spherical only | Any (with appropriate linkage) | Any (density-connected) |
| **Handles outliers** | No (forces every point into a cluster) | No | Yes (label = −1) |
| **k required?** | Yes | No (cut dendrogram) | No |
| **Time complexity** | O(n·k·iter) | O(n² log n) naive | O(n²) naive, O(n log n) with kd-tree |
| **Scales to large n** | Very well | Poorly (O(n²) memory) | Moderate |
| **Primary hyperparams** | k | n_clusters or distance threshold | ε, min_samples |
| **Best use-case** | Well-separated spherical blobs, large datasets | Hierarchical relationships, dendrograms | Arbitrary shapes, anomaly detection, geographic data |

> **Rule of thumb for retail data:** Use K-means first for speed. If you suspect non-spherical clusters or need outlier detection, switch to DBSCAN.


In [ ]:
np.random.seed(42)

# ── Synthetic retail data: TotalSpend vs. Frequency ──────────────────────────
n_customers = 400

# Segment 1: Budget shoppers — low spend, high frequency
seg1_spend = np.random.normal(50, 10, 120)
seg1_freq  = np.random.normal(30, 5, 120)

# Segment 2: Occasional big spenders — high spend, low frequency
seg2_spend = np.random.normal(300, 30, 150)
seg2_freq  = np.random.normal(5, 2, 150)

# Segment 3: Regular mid-tier shoppers — crescent-shaped path
theta = np.linspace(0.2, 2.8, 100)
seg3_spend = 150 + 60*np.cos(theta) + np.random.normal(0, 8, 100)
seg3_freq  = 15  + 12*np.sin(theta) + np.random.normal(0, 2, 100)

# A few outliers: ultra-high spend (potential VIPs or data errors)
outliers_spend = np.array([800, 750, 900, 650, 200])
outliers_freq  = np.array([1,   2,   1,   3,   50])

spend = np.concatenate([seg1_spend, seg2_spend, seg3_spend, outliers_spend])
freq  = np.concatenate([seg1_freq,  seg2_freq,  seg3_freq,  outliers_freq])
X_retail = np.column_stack([spend, freq])

# ── Standardize before DBSCAN ─────────────────────────────────────────────────
scaler = StandardScaler()
X_retail_scaled = scaler.fit_transform(X_retail)

# ── Use k-distance graph to pick eps ─────────────────────────────────────────
sorted_dists_retail = k_distance_graph(X_retail_scaled, k=4)
second_deriv_retail = np.diff(np.diff(sorted_dists_retail))
elbow_retail = np.argmax(second_deriv_retail) + 1
eps_retail   = sorted_dists_retail[elbow_retail]

print(f"Suggested ε for retail data (scaled): {eps_retail:.4f}")

# ── Fit DBSCAN ────────────────────────────────────────────────────────────────
retail_labels = dbscan_scratch(X_retail_scaled, eps=max(eps_retail, 0.3), min_samples=5)

# ── Visualise ─────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: k-distance graph
axes[0].plot(sorted_dists_retail, color='steelblue', linewidth=2)
axes[0].axhline(y=eps_retail, color='tomato', linestyle='--',
                label=f'Elbow ε ≈ {eps_retail:.3f}')
axes[0].set_title('k-Distance Graph — Retail Data', fontsize=12)
axes[0].set_xlabel('Points (sorted)')
axes[0].set_ylabel('4-NN distance')
axes[0].legend()

# Right: DBSCAN result
palette_r = ['#4C72B0', '#DD8452', '#55A868', '#C44E52']
unique_retail = np.unique(retail_labels)
for lbl in unique_retail:
    mask = retail_labels == lbl
    if lbl == -1:
        axes[1].scatter(X_retail[mask, 0], X_retail[mask, 1],
                        c='dimgrey', marker='X', s=100, zorder=5,
                        label=f'Noise / Outliers ({mask.sum()})')
    else:
        axes[1].scatter(X_retail[mask, 0], X_retail[mask, 1],
                        c=palette_r[lbl % len(palette_r)], s=30, alpha=0.7,
                        label=f'Segment {lbl} (n={mask.sum()})')

axes[1].set_title('DBSCAN on Retail Data — Outliers Highlighted', fontsize=12)
axes[1].set_xlabel('Total Spend (£)')
axes[1].set_ylabel('Purchase Frequency')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

n_seg = len(set(retail_labels) - {-1})
n_noise_r = (retail_labels == -1).sum()
print(f"Segments found: {n_seg}  |  Noise (outlier) customers: {n_noise_r}")


In [ ]:
np.random.seed(42)

# ── Verify scratch implementation against sklearn DBSCAN ──────────────────────
eps_v = 0.25
ms_v  = 5

scratch_lbs = dbscan_scratch(X_moons, eps=eps_v, min_samples=ms_v)
sklearn_lbs = SklearnDBSCAN(eps=eps_v, min_samples=ms_v).fit_predict(X_moons)

# Compare cluster counts (label ordering may differ)
def cluster_stats(labels, name):
    n_c = len(set(labels) - {-1})
    n_n = (labels == -1).sum()
    print(f"{name:20s}  clusters={n_c}  noise={n_n}")

cluster_stats(scratch_lbs, "Scratch DBSCAN")
cluster_stats(sklearn_lbs, "sklearn DBSCAN")

# Check that the clustering structure matches (up to label permutation)
# Two points in the same cluster in scratch should be in the same cluster in sklearn
def same_cluster_agreement(lbs_a, lbs_b):
    """Fraction of point-pairs where both agree on same-cluster vs. different-cluster."""
    n = len(lbs_a)
    agree = 0
    total = 0
    # Sample 2000 random pairs for speed
    rng = np.random.default_rng(0)
    pairs = rng.integers(0, n, (2000, 2))
    for i, j in pairs:
        if i == j:
            continue
        same_a = (lbs_a[i] == lbs_a[j]) and (lbs_a[i] != -1)
        same_b = (lbs_b[i] == lbs_b[j]) and (lbs_b[i] != -1)
        agree += int(same_a == same_b)
        total += 1
    return agree / total

agreement = same_cluster_agreement(scratch_lbs, sklearn_lbs)
print(f"\nPairwise cluster-agreement between scratch and sklearn: {agreement:.3f}")
print("(1.0 = perfect agreement on which points share a cluster)")


## DBSCAN Limitations

DBSCAN is powerful but has known failure modes:

### 1. Varying Density Clusters
DBSCAN uses a **single global ε**. If one cluster is dense and another is sparse, no single ε will correctly separate both — it either splits the sparse cluster into noise or merges the dense cluster with a neighbour.

### 2. Computational Cost
- **Naive implementation:** O(n²) time — computing all pairwise distances
- **With a kd-tree or ball-tree** (as sklearn does): O(n log n) for low-dimensional data
- In high dimensions (d > 10), the kd-tree advantage disappears (curse of dimensionality)

### 3. Sensitivity to ε in High Dimensions
In high-dimensional spaces, distances become increasingly uniform — everything looks roughly equally far from everything else. This makes the k-distance elbow very flat, making ε hard to pick.

> **Practical advice:** Run PCA or UMAP first to reduce dimensions, then apply DBSCAN on the low-dimensional representation.


In [ ]:
np.random.seed(42)

# ── Demo: two clusters with very different densities ─────────────────────────
# Dense cluster: tight Gaussian
dense_cluster = np.random.randn(150, 2) * 0.2 + [0, 0]
# Sparse cluster: wide Gaussian
sparse_cluster = np.random.randn(150, 2) * 1.2 + [5, 0]

X_mixed_density = np.vstack([dense_cluster, sparse_cluster])
true_labels = np.array([0]*150 + [1]*150)

# Try a single eps — show the problem
eps_too_small = 0.25   # captures dense cluster but sparse cluster becomes noise
eps_too_large = 1.0    # captures both but may merge them or include background noise

lbs_small = dbscan_scratch(X_mixed_density, eps=eps_too_small, min_samples=5)
lbs_large = dbscan_scratch(X_mixed_density, eps=eps_too_large, min_samples=5)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (lbs, title) in zip(axes, [
    (true_labels,  'True Labels (2 clusters)'),
    (lbs_small,    f'DBSCAN ε={eps_too_small} — too small\n(sparse cluster → noise)'),
    (lbs_large,    f'DBSCAN ε={eps_too_large} — too large\n(clusters may merge)'),
]):
    unique = np.unique(lbs)
    pal = ['#4C72B0', '#DD8452', '#55A868']
    for lbl in unique:
        mask = lbs == lbl
        clr = 'dimgrey' if lbl == -1 else pal[lbl % 3]
        mkr = 'X' if lbl == -1 else 'o'
        ax.scatter(X_mixed_density[mask, 0], X_mixed_density[mask, 1],
                   c=clr, marker=mkr, s=25, alpha=0.7,
                   label='Noise' if lbl == -1 else f'Cluster {lbl}')
    ax.set_title(title, fontsize=10)
    ax.legend(fontsize=8)

fig.suptitle('DBSCAN Fails with Varying-Density Clusters', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

for name, lbs in [('ε=0.25', lbs_small), ('ε=1.0', lbs_large)]:
    n_c = len(set(lbs) - {-1})
    n_n = (lbs == -1).sum()
    print(f"{name}: {n_c} cluster(s), {n_n} noise points")


## HDBSCAN — A Peek at the Solution

**HDBSCAN** (Hierarchical DBSCAN) extends DBSCAN to handle **variable-density clusters**.

Key ideas:
- Instead of a fixed ε, it builds a **hierarchy of clusterings** across all possible ε values
- It extracts the **most stable** clusters from this hierarchy
- Result: clusters with different internal densities are correctly identified

In Python: `from hdbscan import HDBSCAN` or `sklearn.cluster.HDBSCAN` (sklearn ≥ 1.3)

```python
from sklearn.cluster import HDBSCAN
hdb = HDBSCAN(min_cluster_size=10)
hdb.fit(X_mixed_density)
```

HDBSCAN is now the recommended default over plain DBSCAN for most real-world applications. We will not implement it from scratch here — but keep it in your toolkit.


## Self-Check Questions

Test your understanding before moving on.

---

**Q1.** A point p has 3 neighbours within distance ε (including itself), and min_samples = 5. Is p a core, border, or noise point? What happens if p is within ε of a core point q?

<details><summary>Show answer</summary>

p has only 3 neighbours but needs 5 to be a core point, so p is **not a core point**. 
However, if p is within ε of a core point q, then p becomes a **border point** — it joins q's cluster even though it cannot seed a new cluster itself. 
If p is not within ε of any core point, it remains **noise** (label = −1).

</details>

---

**Q2.** You increase ε from 0.3 to 0.8 while keeping min_samples fixed. What do you expect to happen to (a) the number of clusters and (b) the number of noise points?

<details><summary>Show answer</summary>

(a) **Number of clusters decreases** — larger ε connects previously separate regions, merging clusters together. At a large enough ε, everything merges into one cluster. 
(b) **Number of noise points decreases** — with a larger neighbourhood radius, previously isolated points now fall within ε of a core point and become border points.

</details>

---

**Q3.** Why does DBSCAN outperform K-means on the make_moons dataset, even though both use Euclidean distance?

<details><summary>Show answer</summary>

K-means assigns each point to the **nearest centroid**, forcing a Voronoi partition of space into convex regions. The two moon shapes share the same spatial neighbourhood in the middle — a centroid placed there cannot cleanly separate them. 
DBSCAN instead uses **density-connectivity**: two points in the same moon are linked by a chain of core points, while points in different moons cannot be linked without crossing a low-density gap. The algorithm follows the shape of the data rather than imposing a geometric boundary.

</details>


## Key Takeaways

1. **Three point types drive DBSCAN:** core points seed and grow clusters; border points are claimed by adjacent clusters; noise points (label = −1) are genuine outliers.

2. **DBSCAN finds clusters of any shape** by following chains of density-connected core points — no assumption of spherical or convex clusters.

3. **No k required:** the number of clusters is determined by the data and the parameters ε and min_samples.

4. **ε selection:** use the k-distance graph elbow method. Set k = min_samples − 1 and look for the sharp uptick in sorted k-NN distances.

5. **Limitations:** a single global ε struggles with varying-density clusters. For those cases, upgrade to HDBSCAN.

6. **Retail application:** noise points are commercially valuable — they can indicate VIP customers, fraudulent behaviour, or data quality issues worth investigating manually.

---

**Next:** Notebook 6 — Gaussian Mixture Models: when you need soft, probabilistic cluster assignments.
